In [ ]:
Content
1. Explain And Analyse
2. Debugging
3. Temporary Tables
4. Keys
5. Joins

# 1. Explain And Analyse

In [ ]:
# EXPLAIN
	•	Purpose: Shows the query execution plan — how the database intends to execute your SQL.
	•	Tells you:
    	•	Which indexes will be used.
    	•	Whether it will do a full table scan or index scan.
    	•	Join order and join algorithms.
    	•	Estimated number of rows at each step.
            - It returns the rows value is only an estimate from the query planner.
        •	Example: 
            - Output: Seq Scan on orders  (cost=0.00..431.00 rows=21000 width=84)
            - rows=21000 means the planner thinks ~21,000 rows will be scanned, based on the table statistics.
	•	Limitation:
    	•	EXPLAIN by itself shows the planner’s estimation, not the actual execution time.

# EXPLAIN ANALYZE
	•	Executes the query and shows:
    	•	Actual execution times (per step).
    	•	Actual row counts processed.
    	•	Whether estimates matched reality (helps detect bad stats).
	•	Basically: EXPLAIN ANALYZE = EXPLAIN + run the query + measure real performance.        
    •	Example:
        - Output: Seq Scan on orders  (cost=0.00..431.00 rows=21000 width=84)
                                      (actual time=0.050..12.000 rows=20512 loops=1)
        - Estimated rows: 21000
        - Actual rows: 20512

# Usecase
	•	Detect slow steps in the query plan.
	•	Compare estimated vs actual rows — big differences mean statistics might be outdated.
	•	See if an index is being used or if it’s doing a sequential scan.
	•	Tune:
    	•	Indexing strategy
    	•	Join order
    	•	WHERE clauses
    	•	Query rewrite

# Example
Query: EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 42 ORDER BY order_date DESC LIMIT 5;
Output: 
        Limit  (cost=0.43..14.45 rows=5 width=84) (actual time=0.080..0.090 rows=5 loops=1)
          ->  Index Scan using idx_customer_date on orders
                (cost=0.43..1024.56 rows=500 width=84)
                (actual time=0.075..0.085 rows=5 loops=1)
                Index Cond: (customer_id = 42)
        Planning Time: 0.123 ms
        Execution Time: 0.115 ms

📌 Caution: EXPLAIN ANALYZE runs the query, so if it’s a huge DELETE or UPDATE, it will modify data.                                                             

# 2. Debugging

In [ ]:
1. Break queries into parts
    - Run sub-queries and individual joins seperately.
2. Check each filter
    - If you have 5 filter on a query per table, first remove all filters and then add filters one by one.
3. Validate Joins
    - Using a wrong join type is a common mistake.
    - Run query on individual table and check results, and then, add join and check results.
4. Check for Nulls
6. Check Aggregation
    - For GROUP BY, verify the grouping keys.
    - In group by queries, adding an additional helper column might help.
7. Log Intermediate Results
    - You can run each CTE individually to see intermediate steps.
    - Query
            WITH filtered_orders AS (
                SELECT * FROM orders WHERE order_date >= '2025-08-01'
            )
            SELECT * FROM filtered_orders WHERE total_amount > 100;

In [ ]:
#SQL Profiler
- A graphical tool, It allows you to capture, monitor, and analyze events happening in the SQL Server database in real time.

- Purpose
	1.	Query Performance Troubleshooting
    	•	Identify slow or expensive queries.
    	•	Find missing indexes or inefficient query patterns.
	2.	Activity Monitoring
    	•	Track who is connecting to the DB and from where.
    	•	Monitor what queries are being run (helpful for security audits).
	3.	Debugging
    	•	See actual queries sent by an application (sometimes app ORM changes queries unexpectedly).
	4.	Workload Analysis
    	•	Capture a workload to replay on a test server for performance testing.
	5.	Audit & Security
    	•	Track unauthorized access or suspicious activity.

UseCase (A web app is slow during peak hours)
    1.	Open SQL Profiler.
	2.	Create a trace filtering for:
    	•	Duration > 1000ms
    	•	Database = ProductionDB
	3.	Start the trace during peak traffic.
	4.	Look for:
    	•	Queries with high CPU/Reads
    	•	Missing index warnings
    	•	Repeated queries from the same app call
	5.	Optimize those queries or indexes.

# 3. Temporary Tables

In [ ]:
+--------------------+---------------------------------------------+-----------------------------+---------------------------+-------------------------+------------------------+-------------------------------+---------------------------------------------+
| Concept            | What it is                                  | Lifetime / Scope            | Physically stored?        | Reusable in same stmt?  | Indexes allowed?       | Persists across statements?   | Typical use cases                             |
+--------------------+---------------------------------------------+-----------------------------+---------------------------+-------------------------+------------------------+-------------------------------+---------------------------------------------+
| CTE (non-recursive)| Named, inline result set (WITH ...)         | Only during that statement  | No (logical/optimizer)    | Yes (multiple refs)     | No                     | No                            | Readability, reuse subquery, stepwise logic  |
| CTE (recursive)    | CTE that references itself to walk hierarchies| Only during that statement | No (computed on the fly)  | Yes                     | No                     | No                            | Trees/graphs, bill-of-materials, calendars   |
| Derived table      | Subquery in FROM (...)                       | Only during that statement  | No (inline)               | No (unless repeated)    | No                     | No                            | Quick, one-off inline transformation         |
| Temp table         | Real table in temp space (#temp, TEMP TABLE) | Session (or txn)            | Yes (tempdb/disk/RAM)     | Yes (across statements) | Yes (can add indexes)  | Yes (within session/txn)       | Complex multi-step ETL, big intermediates    |
| Table variable*    | Lightweight table variable (@t in SQL Server)| Batch/scope-limited         | Yes (spills to tempdb)    | Yes (within scope)      | No    (engine-specific)| No (ends with scope)         | Small sets, procedural T-SQL                 |
| View               | Saved SELECT (logical table)                 | Permanent (until dropped)   | No (not data)             | Yes (any statement)     | No (on view itself)**  | Yes                           | Reuse logic, security, abstraction           |
| Materialized view  | View whose result is stored (indexed view)   | Permanent (until dropped)   | Yes (precomputed data)    | Yes                     | Yes (on the MV)        | Yes                           | Speed up aggregates/joins, caching results   |
+--------------------+---------------------------------------------+-----------------------------+---------------------------+-------------------------+------------------------+-------------------------------+---------------------------------------------+
* Table variable is SQL Server–specific; other engines have roughly analogous constructs (e.g., PL/pgSQL arrays/temporary tables).
** You can index underlying base tables; SQL Server “indexed views” and Postgres “materialized views” can have indexes.

    
Object Type        | Scope           | Lifetime                  | Visible Across Sessions? | Persist After Restart?
---------------------------------------------------------------------------------------------------------------
CTE                | Query           | Single query              | NO                       | NO
Derived Table      | Query           | Single query              | NO                       | NO
Temp Table (#)     | Session         | Until session ends        | NO                       | NO   (Can add index)
Global Temp (##)   | All sessions    | Until last session ends   | YES                      | NO
View               | Database        | Permanent                 | YES                      | YES
Materialized View  | Database        | Permanent (stored data)   | YES                      | YES   (Can add index)


# Cheet Sheet
1. CTE:               Scope (within the query),    Can be used multiple times.                    Ex: {With CTE_NAME as (select query)}
2. Derived Table:     Scope (within the query),    Can not be used multiple times.                Ex: {(select query) As d}
3. Temp Table:        Scope (Untill session ends), Used Multiple times - normal table with index  Ex:  Create TEMPORARY table Studnet... 
4. View:              Scope (Permanenet) - Just a query, Execute each time when used.             Ex: CREATE VIEW v_active_customers AS ()
5. Materialized View: Scope (Permanenet) - Query with result, 

    
Choosing the right tool (quick heuristics)
	•	Need readability + reuse within one statement? → CTE
	•	One-off inline transform? → Derived table
	•	Large intermediate you’ll touch multiple times / want indexes? → Temp table
	•	Reusable abstraction across many queries? → View
	•	Precompute/accelerate expensive queries? → Materialized view / Indexed view
	•	Tiny procedural sets in T-SQL? → Table variable




Performance tips
	•	CTE vs derived table: usually equivalent; CTE improves readability. Don’t assume CTE materializes; check your engine’s optimizer.
	•	Temp tables: add indexes if you join/aggregate; can drastically improve performance on large intermediates.
	•	Views: watch for nested views; they can hide expensive plans. Profile with actual execution plans.
	•	Materialized views: great for speed, but plan for refresh (schedule or incremental maintenance).

In [ ]:
1. CTEs / With Query
    - CTE = temporary named table result (CTE defined before your main query)
    - Common table Expression is a temporary named result set that you define within a SQL statement.
	- Exists only for the duration of that query execution (not stored in the DB permanently). Can we re-used multiple times in same query.
    - Created only in-memory, once a query is completed, CTEs are removed from memory.
	- Is defined before your main SELECT, INSERT, UPDATE, or DELETE
	- Makes queries more readable and modular
    -  No, you cannot create an index directly on a CTE.
    - usercase:
        - 	Debugging: You can run the CTE separately to verify partial results.
    	-	Avoid duplication: Reduces copy-pasting of subqueries, same CTE can be used multiple time in a query.
        -   It can be used like a sub-query.


# Example Query
WITH cte_name AS (
    SELECT column1, column2
    FROM table
    WHERE condition
)
SELECT *
FROM cte_name
WHERE column1 > 100;

SELECT *
FROM cte_name
WHERE column1 > 100;

2. Derived Table
    - It is inline subquery (inside query) and it can not be re-used in the same query.
    - A derived table is not stored in the database — it’s just a logical, inline result set created by the SQL engine while executing a query.
    - Scope: Only duration of the single query. It disappears immediately after the outer query finishes the execution.
        - If we re-run the query, it built from scratch.
    - What: A subquery used inline in FROM. It’s not named for reuse (unless you repeat it).
    - Query:
            SELECT d.customer_id, COUNT(*) AS orders_cnt
            FROM (
              SELECT customer_id
              FROM orders
              WHERE order_date >= CURRENT_DATE - INTERVAL '30 days'
            ) AS d
            GROUP BY d.customer_id;

3. Temporary Tables
    - Temporary tables are real physical tables (just created in a special temporary storage area like tempdb in SQL Server or the session’s temporary tablespace in PostgreSQL/MySQL).
    - Scope
        - 	Session-scoped temporary table → Automatically dropped when your session/connection closes.
    	-	Transaction-scoped temporary table (PostgreSQL has this option) → Dropped at end of the transaction.
        -   If session crashes or time out, DB automatically cleans up.
        -   In a connection pool application as well, each connection pool will only see its own temp tables.
        -   via Query: DROP TABLE tmp_table;  
    - Update operation
        - Add Index, Alter structure, Join with permanent tables
    - query to create temp table
        - SQL Server: INSERT INTO #tmp_recent_orders (order_id, customer_id) VALUES (101, 1);
        - CREATE TABLE #temp (id INT); and   CREATE INDEX idx_temp_id ON #temp(id);
        - sql/ Postgress: 
            CREATE TEMPORARY TABLE temp_users (
                id SERIAL PRIMARY KEY,
                username TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );


4. Table Variable
    - a variable that can store tabular data in memory (and sometimes tempdb) within the scope of a batch, stored procedure, or function.
    - It behaves like a temporary table in that you can store rows and query them, but:
    	•	It’s declared with DECLARE @varName TABLE(...)
    	•	It’s scoped to the batch/procedure/function that declared it
    	•	It’s treated more like a variable than a fully-fledged table
    - Index
        •	You can define PRIMARY KEY or UNIQUE constraints when declaring → these create indexes internally.
    	•	You cannot add non-clustered indexes after declaration (unlike temp tables).
    - Performance
        •	Good for small datasets (few hundred rows).
    	•	Bad for large joins/aggregations because, No statistics = optimizer often assumes 1 row (bad plan choice).
    	•	Can’t create extra indexes after declaration.
    - When not to use:
        •	Working with large datasets.
    	•	You need indexes beyond PK/Unique.
    - Query

            BEGIN
                DECLARE @t TABLE (id INT);
                INSERT INTO @t VALUES (1);
                SELECT * FROM @t; -- Works
            END
            SELECT * FROM @t; -- ERROR: @t not declared here


5. View
    - A saved SELECT (logical table). No data stored (except security/perms); every query against a view re-executes its definition against base tables.
    - each use runs the underlying query; can hide complexity (be mindful of performance).
    - A view in SQL Server is a permanent database object (like a table, function, or stored procedure).
    	•	It is stored in the system catalog (sys.objects, sys.views) inside the database.
    	•	Once created, it can be used in any session/connection to that database (subject to permissions).
    	•	Unlike CTEs or temp tables, it is not tied to a session or query execution.

    - Query:
            CREATE VIEW v_active_customers AS
                SELECT id, name, email FROM customers WHERE status = 'active';
                
                SELECT * FROM v_active_customers WHERE name LIKE 'S%';

6. Materialized View/ Indexed View (in sql server)
    - A materialized view is like a normal SQL view (predefined query) but with the result set stored physically on disk, not computed each time you query it.
    	•	In PostgreSQL, Oracle, MySQL → called Materialized View.
    	•	In SQL Server → called Indexed View (because the storage happens when you create a clustered index on the view).
    - It’s basically a cached table that automatically (or manually) gets refreshed from the underlying base tables.
    - Scope: These are stored permanently in the disk, and needs to drop explicitely.
                - Query to drop: DROP MATERIALIZED VIEW sales_summary;
    - Types
            a. Automatic Refresh Materialized Views: Data updates in base tables → MV/Indexed View is updated in-place.
            b. Manual Refresh Materialized Views: Old data stays until you explicitly REFRESH MATERIALIZED VIEW.
    - Use Cases
        ✅ Best for:
        	•	Frequently executed complex joins & aggregations
        	•	Reporting dashboards where data changes infrequently
        	•	Precomputed analytics for BI tools
        
        🚫 Avoid if:
        	•	Data changes constantly and you can’t tolerate slower writes
        	•	Queries rarely repeat the same expensive computation

    - SQL server Query:
                        CREATE MATERIALIZED VIEW SalesSummary
                                WITH SCHEMABINDING
                                AS
                                SELECT region, SUM(amount) AS total_sales
                                FROM dbo.Sales
                                GROUP BY region;
                                
                        CREATE UNIQUE CLUSTERED INDEX IX_SalesSummary ON SalesSummary(region);


7. Cache Table (just a design pattern)
 	•	A physical normal table in the database designed to hold data that Is expensive to compute. Where we do not want to compute again and again, we just create a new table that stores the summary of our queries.

# 4. Keys

In [ ]:
+----------------+-------------------------------------+------------------------------------+
| Key Type       | Purpose                             | Example                            |
+----------------+-------------------------------------+------------------------------------+
| Candidate Key  | All possible unique identifiers     | emp_id, email                      | All possible choices for Primary Key, each set must have minimal keys.
| Primary Key    | Chosen unique identifier            | emp_id                             |
| Alternate Key  | Candidate key not chosen as PK      | email                              |
| Foreign Key    | Link to another table's PK          | emp_id in Orders                   |
| Composite Key  | PK using multiple columns           | (student_id, course_id)            |
| Super Key      | Any unique column set               | (emp_id, email)                    |
| Unique Key     | Enforce uniqueness constraint       | username     -Null Allowed         | Nulls are allowed, two unknown values are not considered equal. So multiple NULLs do not violate uniqueness.
| Surrogate Key  | Artificial, system-generated key    | auto-increment id                  |
| Natural Key    | Real-world identifier               | passport_number                    |
| Compound Key   | Multiple columns, not always PK     | (order_id, product_id)             |
+----------------+-------------------------------------+------------------------------------+

Super Key -> Candidate Key (subset, minimal) -> Primary Key/ Composite Primary Key (choosen one)

Alternate key = { Candidate key } - { primary key }     

In [ ]:

1.  Super Key (unique + Extra columns)
    - Any set of columns that uniquely identifies a row (includes candidate keys + other columns with null values).
    - Can have extra unnecessary columns in addition to what’s needed for uniqueness.
	- Purpose: The superset of candidate keys.
	- Example:
         -  {ID}, {SSN}, {ID, Name},
         -   {ID, SSN}, (ID, Phone},
         -   {Name, Phone}, {ID, Email},
         -   {Name, SSN, Phone},
         -   {Name, Email },    # Email may have null.
         -   {ID, SSN, Phone} 
         - ❌ Name - As it will have repeated values, and it can not identify a row uniquely.


2. Candidate Key (minimum set of column(s) of super key)
	- A minimal set of column(s) that can uniquely identify a row in a table. removing any column will break uniqueness.
    - Filter Super Key  set with Minimal column called Candidate key.
    - This is also a set of columns.
    - Can be single-column or multi-column (composite).
	-	Purpose: Represents all possible choices for the Primary Key.
	-	Rules:
    	-	Must have unique values for each row.
    	-	Cannot contain NULLs.
    - if ID is already considered, then any other set should not have Id.
    - Example:
        - {ID}, 
        - {SSN},
        - {Email}                            
        - {Phone} 
                            
2. Primary Key (one column with not null, out of candiate key set)
    - It is the choosen Candidate Key with single column, that  Uniquely identify row, and should not contain null.
	-	Purpose: Enforce entity integrity.
	-	The database automatically creates a clustered index (in most SQL DBs) on it.
	-	Rules:
    	-	Only one primary key per table.
    	-	Cannot contain NULLs.
    - Example: {Id}, {SSN}
            - {Email} ❌ Name - As it may have null.
                            
3. Composite Primary key (multiple column combination with not null, out of candiate key set)
    - It is the choosen Candidate Key with multiple column, that  Uniquely identify row, and should not contain null in any column.
    - All composite primary keys are composite keys, but not all composite keys are primary keys.                        
	- Purpose:
            - When no single column is unique by itself.
            - It uniquely identifies rows.
        	- It is used as the main reference for foreign keys.
	-	The database automatically creates a clustered index (in most SQL DBs) on it.
    - Example:
                CREATE TABLE Enrollment (
            student_id INT,
            course_id INT,
            PRIMARY KEY (student_id, course_id) -- composite primary key
        );                            

4. Alternate Key
    - In the set of candiate key, all sets other than the primary key.
    - Example:
        - If Id is choosen as primary key, then {SSN}, {Email}, {Phone} are alternate key




Composite Key
     - A key made up of two or more columns used together to uniquely identify a row.
     - Purpose: Sometimes no single column alone can uniquely identify data. In such cases, we combine multiple columns.
     - Example:
        - (student_id, course_id) # in case if one student takes multiple courses.



                            

# 5. Joins

In [ ]:
+-----------------+-----------------------------------+--------------------------------------------------------------+
| Join Type       | What It Does                      | Example Use Case                                             |
+-----------------+-----------------------------------+--------------------------------------------------------------+
| INNER JOIN      | Returns only matching rows in     | Find customers who have placed at least one order.           |
|                 | both tables                       |                                                              |
| LEFT JOIN       | All rows from left table +        | Find all customers, and their orders if they exist.          |
|                 | matching from right table         |                                                              |
| RIGHT JOIN      | All rows from right table +       | Find all orders, and the customers who placed them.          |
|                 | matching from left table          |                                                              |
| FULL OUTER JOIN | All rows from both tables,        | See all customers and all orders, even if they don't match.  |
|                 | matched where possible            |                                                              |
| CROSS JOIN      | Cartesian product (every row      | Generate all possible combinations of two lists.             |
|                 | from left with every row in right)|                                                              |
| SELF JOIN       | Join table with itself            | Find employees who report to the same manager.               |
+-----------------+-----------------------------------+--------------------------------------------------------------+

1. Inner Join
    - Rows that have matching values in both tables.
    - Example: If a customer has no orders, they won’t appear.
    - Query
        SELECT c.customer_id, c.name, o.order_id
        FROM customers c
        INNER JOIN orders o       # Or just JOIN
            ON c.customer_id = o.customer_id;

2. Left Join
    - All rows from left table, matched rows from right table.
    -  Good for finding customers without orders (WHERE o.order_id IS NULL).
    - Query
            SELECT c.customer_id, c.name, o.order_id
            FROM customers c
            LEFT JOIN orders o
                ON c.customer_id = o.customer_id;

3. Right Join
    - All rows from right table, matched rows from left table.
    - Query: RIGHT JOIN orders o

4. FULL OUTER JOIN
    - All rows from both tables, match where possible.
    - Non matching columns get NULL
    - Query
            SELECT c.customer_id, c.name, o.order_id
            FROM customers c
            FULL OUTER JOIN orders o
            ON c.customer_id = o.customer_id;

5. CROSS JOIN / Cartesian product
    - Cartesian product — every row from left combined with every row from right.
    - When to use: Combinations, testing, generating datasets.
    - If colors has 5 rows and sizes has 3 rows → 15 rows output.
    - Query
            SELECT a.color, b.size
            FROM colors a
            CROSS JOIN sizes b;
6. Self JOIN
    - Matches rows in the same table to each other.
    - Query
            SELECT e1.employee_id, e1.name, e2.name AS manager_name
            FROM employees e1
            LEFT JOIN employees e2
                ON e1.manager_id = e2.employee_id;

7. Natural Join
    - Automatically joins tables on all columns with the same name.
    - Query:
            SELECT *
            FROM customers
            NATURAL JOIN orders;